# 00 — Run everything (single resumable pipeline)

One notebook, in dependency order. **Every stage skips itself if its output already exists.**
When a Kaggle session times out: **Save Version** -> **Add Input -> that output** -> re-run from
the top. It resumes.

## Session plan (fits Kaggle's 9 h session / 30 h weekly GPU quota)

| session | stages | est. | ends with |
|---|---|---|---|
| **1** | 0-2: setup, calibration (no pixel ladder), BRANCH decision | **~5 h** | a pinned probe config, or a hard stop at BRANCH B |
| **2** | 3-5: re-sweep 3 cells, contract check, confirmatory H1-H4 | **~4-5 h** | the first real verdicts |
| **3** | 6-8: displacement (random, trained), link test, pixel diagnostic | **~4 h** | the A11 results |

Total ~13-14 h. Run notebooks `04` then `05` (recipe control, ~48 h of *training*) separately and
in parallel — they gate nothing here.

## Why the pixel ladder is not in the pin pass

Raw-pixel features are 12288-d, so `mlp_large`/`mlp_deep` cost ~77 GFLOP per step against ~3-4
for the 512-d backbone: **~25x per fit**. Swept across all 8 (regime x size) combinations that is
~24 h — longer than a Kaggle session, in a process that would restart from zero each time. It
would never finish.

A4 §b makes the pixel ladder **diagnostic-only**: *"Enters no decision rule and no confirmatory
family."* The A10 §b gate that decides BRANCH A/B reads `encoder_role == "random"` only. So the
pin pass runs `--no-pixel-reference`, and stage 2b runs the pixel ladder **once, at the pinned
config**, which is all the diagnostic needs — ~1.5 h instead of ~24 h.

## Two things this notebook deliberately will NOT do

**It stops at BRANCH B.** If no interpolation config clears the A10 §b all-factor headroom gate,
stage 2 raises and refuses to continue. Moving to the composition regime **changes the estimand**
from lattice interpolation to compositional generalization, and that decision is yours to make and
to file as a dated amendment — automating past it is exactly the researcher degree of freedom the
amendment discipline exists to remove. The draft is at
`lab-notebook/drafts/A12-composition-regime-DRAFT.md`; stage 2 prints the trigger numbers, and once
A12 is filed you set `REGIME = "composition"` and re-run.

**It does not run the recipe control.** ~4 GPU-hours x 12 seeds of *training*, gates nothing else,
exploratory under A7 §c. Notebooks `04` (1-seed pilot) then `05` (12 seeds).

## 0. Setup

In [ ]:
!nvidia-smi

In [ ]:
import os
REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
%cd {REPO_DIR}

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

### `sh()` — a shell helper that actually fails

IPython's `!cmd` swallows the exit code: a stage that dies leaves the notebook running and the
next stage reads stale artifacts. In a seven-stage pipeline that is how a void number reaches a
verdict table. Everything below goes through `sh()`, which raises on a non-zero exit.

In [ ]:
import subprocess, sys

REPO_DIR = "/kaggle/working/probe-capacity-invariance"

def sh(cmd: str) -> None:
    """Run one shell command in the repo; RAISE on failure so a dead stage stops the run."""
    print("$", " ".join(cmd.split()), flush=True)
    r = subprocess.run(cmd, shell=True, cwd=REPO_DIR)
    if r.returncode != 0:
        raise SystemExit(f"stage command FAILED (exit {r.returncode}) — fix it before "
                         f"continuing; later stages would read stale artifacts.\n  {cmd}")

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| devices:", torch.cuda.device_count())

In [ ]:
sh("python -m src.data.shapes3d --download --build-cache")
sh("python -m src.data.dsprites --download --build-cache")

In [ ]:
# Restore prior outputs + encoder checkpoints so a timed-out session resumes.
import shutil
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance"); INPUT = Path("/kaggle/input")
restored = 0
for src in list(INPUT.glob("*/results")) + list(INPUT.glob("*/probe-capacity-invariance/results")):
    for f in src.rglob("*"):
        if f.is_file() and f.suffix in (".npz", ".json", ".jsonl", ".pt"):
            dst = REPO / "results" / f.relative_to(src)
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst); restored += 1
print(f"restored {restored} prior file(s)")

## Preflight — fail loudly on stale code

Burning a session on a pre-A10 checkout is the expensive mistake. Also verifies the A11 blind
guard actually fires: a guard that does not fire is not a guard.

In [ ]:
import argparse, inspect
from src.probes import ladder, instrument, instrument_calibration, hypotheses, run_sweep
from src.probes import displacement as D
from src.data import splits

src_cal, src_hyp = inspect.getsource(instrument_calibration.run), inspect.getsource(hypotheses.analyze_cell)
checks = [
    ("A9  step budget pinned",        getattr(ladder, "DEFAULT_STEPS", None) == 1000),
    ("A10a target standardization",   hasattr(ladder, "_standardize_targets")),
    ("A10a cache version tag",        getattr(ladder, "INSTRUMENT_VERSION", None) == "a10"),
    ("A10b all-factor headroom gate", "all_factor_headroom_ok" in src_cal),
    ("A10c epsilon_D",                hasattr(instrument, "epsilon_d")),
    ("A10c two-sided flip",           hasattr(instrument, "deficit_flip_count")),
    ("A10c H3 keys on eps_invariant", "eps_invariant" in src_hyp),
    ("A11  displacement module",      hasattr(D, "epsilon_m") and hasattr(D, "monotonicity")),
    ("A12  sweep supports regime",    "--regime" in inspect.getsource(run_sweep._main)),
    ("compositional split available", hasattr(splits, "make_combination_holdout_splits")),
]
for n, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {n}")
assert all(ok for _, ok in checks), "stale checkout — git pull above, then restart the kernel"

try:
    D.run(argparse.Namespace(trained_encoders=["x/backbone.pt"], amendment=None))
    raise AssertionError("A11 BLIND GUARD DID NOT FIRE")
except SystemExit as e:
    assert "BLIND GUARD" in str(e)
    print("  PASS  A11 blind guard refuses a trained checkpoint without --amendment")
print("\npreflight OK")

## Stage 1 — all-factor recalibration (BLIND-SAFE)

A9's pin was validated on the categorical **shape** anchor alone, and the anchor is not
representative: at n=40000 its top-rung floor is 0.9858 while `floor_hue` and `wall_hue` reach
0.9982 and 0.9986. A pin that leaves every **targeted** readout saturated measures nothing, so
A10 §b scores the floor on *every* factor. Extraction dominates the cost; skips if already run.

In [ ]:
import json
from pathlib import Path

# A restored calibration file may PREDATE A10 (b) and carry no all-factor headroom fields.
# Skipping on mere existence would let stage 2 report a false BRANCH B whose real cause is a
# stale artifact rather than a saturated instrument.
def is_a10(path: Path) -> bool:
    rows = json.loads(path.read_text()).get("results", [])
    return bool(rows) and "all_factor_headroom_ok" in rows[0]

for ds, partner, nh in (("shapes3d", "orientation", 3), ("dsprites", "scale", 2)):
    out = Path(f"results/calibration/calibration_{ds}.json")
    if out.exists() and is_a10(out) and json.loads(out.read_text()).get("recommended_config") is not None:
        print(f"stage 1 [{ds}]: SKIP (A10 output complete)"); continue
    if out.exists() and not is_a10(out):
        stale = out.with_suffix(".pre_a10.json")
        out.rename(stale)
        print(f"stage 1 [{ds}]: existing output PREDATES A10 (b) — moved to {stale.name}")
    # --no-pixel-reference: the 12288-d pixel ladder is ~25x per fit and A4 (b) makes it
    # diagnostic-only. It runs once at the pinned config in stage 2b instead.
    # --resume: rows are checkpointed per (regime, size, steps, role) so a killed session
    # continues rather than restarting.
    sh(f"python -m src.probes.instrument_calibration "
       f"--dataset {ds} --probe-train-sizes 2000 5000 10000 40000 "
       f"--probe-steps 1000 4000 --partner-factor {partner} --n-holdout-partner {nh} "
       f"--random-seed 0 1 2 --no-pixel-reference --resume "
       f"--device cuda --num-workers 2 --out-root results/calibration")

## Stage 2 — the BRANCH decision

A config is pinnable only if it clears **all three** gates on **both** datasets: A8 §c (anchor has
top-rung headroom), **A10 §b (no factor at all saturated at the top rung)**, and A8 §e (the Adam
linear rung agrees with the convex solver within 0.02, so `Delta_G` is a capacity difference and
not an optimization one).

Prints, per dataset and per candidate config: the top-rung floor for **every** factor,
`worst_top_rung_floor`, `saturated_factors_at_top`, and the linear-rung gap.

In [ ]:
import json
from pathlib import Path

# Set to "composition" ONLY after filing A12. Leave as "interpolation" otherwise.
REGIME = "interpolation"

runs, clearing = {}, {}
for ds in ("shapes3d", "dsprites"):
    p = Path(f"results/calibration/calibration_{ds}.json")
    if not p.exists():
        print(f"{ds}: NOT RUN"); continue
    runs[ds] = r = json.loads(p.read_text())
    SAT = r.get("saturation_level", 0.90)
    print(f"\n=== {ds} ===  partner={r.get('composition_partner')}  saturation={SAT}  "
          f"tol={r.get('gap_tolerance')}")
    for x in [y for y in r["results"] if y["encoder_role"] == "random"]:
        gates = []
        if x["saturated_at_top"]:             gates.append("A8c-anchor-saturated")
        if x.get("saturated_factors_at_top"): gates.append("A10b-saturated:" + ",".join(x["saturated_factors_at_top"]))
        if x["capacity_axis_flag"]:           gates.append(f"A8e-lin-gap={x['linear_rung_gap']:+.4f}")
        print(f"\n  {x['regime']:14s} n={x['probe_train_size']:<6d} steps={x['probe_steps']:<5d} "
              f"worst_top_rung_floor={x.get('worst_top_rung_floor', float('nan')):.4f} "
              f"lin_gap={x['linear_rung_gap']:+.4f} all_factor_ok={x.get('all_factor_headroom_ok')}")
        for name, vals in sorted(x.get("factor_floor_mean", {}).items()):
            print(f"      {name:14s} top-rung floor {vals[-1]:.4f}"
                  + ("   <- SATURATED" if vals[-1] >= SAT else ""))
        print(f"      -> {'CLEARS ALL GATES' if not gates else ' | '.join(gates)}")
    clearing[ds] = [x for x in r["results"] if x["encoder_role"] == "random"
                    and x["regime"] == REGIME and not x["saturated_at_top"]
                    and not x["capacity_axis_flag"] and x.get("all_factor_headroom_ok")]
    print(f"\n  --> {REGIME} configs clearing the ALL-FACTOR gate: {len(clearing[ds])}")

assert runs, "stage 1 has not produced any calibration output"
if not all(clearing.get(ds) for ds in runs):
    raise SystemExit(
        "\nBRANCH B — no " + REGIME + " config clears the A10 (b) all-factor gate.\n"
        "STOP. This is a decision for you, not for the notebook.\n"
        "  1. File lab-notebook/drafts/A12-composition-regime-DRAFT.md as a dated amendment,\n"
        "     filling in the trigger numbers printed above.\n"
        "  2. State explicitly that the ESTIMAND changes from lattice interpolation to\n"
        "     compositional generalization, and that no number under the new regime is\n"
        "     comparable to any interpolation number in A4/A8/A10.\n"
        "  3. Set REGIME = 'composition' in this cell and re-run.\n"
        "This must happen BEFORE any trained-encoder targeted-factor value is computed.")

# Pin: prefer the LARGEST probe-train size that clears, then the smallest step budget.
pin = min((x for ds in runs for x in clearing[ds]),
          key=lambda x: (-x["probe_train_size"], x["probe_steps"]))
PROBE_TRAIN, PROBE_STEPS = pin["probe_train_size"], pin["probe_steps"]
SEEDS = " ".join(str(i) for i in range(12))
print(f"\nBRANCH A — pinned: regime={REGIME} probe_train={PROBE_TRAIN} steps={PROBE_STEPS}")

## Stage 2b — raw-pixel reference at the pinned config (diagnostic, A4 §b)

The pixel ladder quantifies what the **input alone** affords each probe family, which is what
contextualizes the random-encoder floor (a random CNN is a function of pixels). It enters no
decision rule and no confirmatory family, so it only needs to run at the configuration actually
pinned — one row instead of eight, ~1.5 h instead of ~24 h.

Safe to skip if you are short on quota; nothing downstream reads it.

In [ ]:
import json
from pathlib import Path

RUN_PIXEL_DIAGNOSTIC = True   # set False to skip; nothing downstream depends on it

if RUN_PIXEL_DIAGNOSTIC:
    for ds, partner, nh in (("shapes3d", "orientation", 3), ("dsprites", "scale", 2)):
        p = Path(f"results/calibration/pixel_{ds}.json")
        if p.exists():
            print(f"stage 2b [{ds}]: SKIP (exists)"); continue
        comp = "--composition" if REGIME == "composition" else "--no-composition"
        sh(f"python -m src.probes.instrument_calibration "
           f"--dataset {ds} --probe-train-sizes {PROBE_TRAIN} --probe-steps {PROBE_STEPS} "
           f"--partner-factor {partner} --n-holdout-partner {nh} {comp} "
           f"--random-seed 0 --pixel-reference --resume "
           f"--device cuda --num-workers 2 --out-root results/calibration/_pixel")
        src = Path(f"results/calibration/_pixel/calibration_{ds}.json")
        if src.exists():
            rows = [r for r in json.loads(src.read_text())["results"]
                    if r["encoder_role"] == "pixels"]
            p.parent.mkdir(parents=True, exist_ok=True)
            p.write_text(json.dumps({"kind": "pixel_reference_ladder (A4 §b, diagnostic only)",
                                     "dataset": ds, "pinned": {"regime": REGIME,
                                     "probe_train": PROBE_TRAIN, "probe_steps": PROBE_STEPS},
                                     "results": rows}, indent=2))
            for r in rows:
                print(f"  {ds} pixels  worst_top_rung_floor="
                      f"{r.get('worst_top_rung_floor', float('nan')):.4f}  "
                      f"saturated={r.get('saturated_factors_at_top') or 'none'}")
else:
    print("stage 2b: skipped by flag")

## Stage 3 — re-sweep the three realized cells

**A10 (a): every continuous entry in the existing stacks is VOID.** They were computed with the
unstandardized-target ladder. The resume cache is keyed on `INSTRUMENT_VERSION` so a pre-A10 cache
cannot be reloaded at a matching `(n, steps)`, but any restored `stacks.npz` is deleted here so a
stale stack cannot be mistaken for a fresh one.

`control_strong` is gate-exempt by design (A2): a minimal-augmentation encoder is *expected* to
sit at the random floor, so 0/12 passing is the intended datum, not a failure.

In [ ]:
from pathlib import Path

CELLS = (("color_strong",    "shapes3d", "color",    "results/encoders/color_strong_seed*/backbone.pt"),
         ("control_strong",  "shapes3d", "control",  "results/encoders/control_strong_seed*/backbone.pt"),
         ("position_strong", "dsprites", "position", "results/encoders/position_strong_seed*/backbone.pt"))

for cell, _, _, pat in CELLS:
    found = sorted(Path().glob(pat))
    print(f"  {cell:16s} {len(found):2d} checkpoint(s)")
    assert len(found) >= 10, f"{cell}: only {len(found)} encoders (<10 = under-powered)"
    st = Path("results/probes") / cell / "stacks.npz"
    meta = Path("results/probes") / cell / "meta.json"
    if st.exists():
        import json
        m = json.loads(meta.read_text()) if meta.exists() else {}
        fresh = (m.get("probe_train_size") == PROBE_TRAIN and m.get("probe_steps") == PROBE_STEPS
                 and m.get("probe_regime", "interpolation") == REGIME)
        if fresh:
            print(f"    stack already at the pinned config — kept")
        else:
            print(f"    VOID/stale stack (n={m.get('probe_train_size')} "
                  f"steps={m.get('probe_steps')} regime={m.get('probe_regime','interpolation')}) — removing")
            st.unlink()

In [ ]:
from pathlib import Path

PARTNER = {"shapes3d": "orientation", "dsprites": "scale"}
NHOLD   = {"shapes3d": 3, "dsprites": 2}

for cell, ds, cond, pat in CELLS:
    if (Path("results/probes") / cell / "stacks.npz").exists():
        print(f"stage 3 [{cell}]: SKIP (already at pinned config)"); continue
    extra = (f"--regime {REGIME} --partner-factor {PARTNER[ds]} "
             f"--n-holdout-partner {NHOLD[ds]} ") if REGIME == "composition" else ""
    sh(f"python -m src.probes.run_sweep --config configs/probe/ladder.yaml "
       f"--dataset {ds} --condition {cond} --strength strong "
       f"--encoders {pat} --random-seed {SEEDS} "
       f"--subsample {PROBE_TRAIN} --probe-steps {PROBE_STEPS} {extra}"
       f"--device cuda --num-workers 2 --resume --out-root results/probes")

## Stage 4 — contract check

Every cell must carry the A8 §d random-projector floor, checkpoint provenance, and the pinned
probe config. A cell that fails here is not usable by stage 5.

In [ ]:
import json
import numpy as np
from pathlib import Path

ok = True
for cell, *_ in CELLS:
    d = Path("results/probes") / cell
    if not (d / "stacks.npz").exists():
        print(f"{cell:16s} NOT SWEPT"); ok = False; continue
    z = np.load(d / "stacks.npz"); m = json.loads((d / "meta.json").read_text())
    bad = []
    if "random_projector" not in z.files: bad.append("no random_projector (A8 §d)")
    if not m.get("encoder_ckpts"):        bad.append("no checkpoint provenance")
    if m.get("probe_train_size") != PROBE_TRAIN: bad.append(f"n={m.get('probe_train_size')}")
    if m.get("probe_steps") != PROBE_STEPS:      bad.append(f"steps={m.get('probe_steps')}")
    if m.get("probe_regime", "interpolation") != REGIME: bad.append(f"regime={m.get('probe_regime')}")
    gate = m.get("quality_gate", {})
    print(f"{cell:16s} {'OK' if not bad else 'PROBLEM: ' + ', '.join(bad)} | "
          f"gate {gate.get('n_passed')}/{gate.get('n_encoders')} | "
          f"trained {z['trained'].shape} random {z['random'].shape}")
    ok &= not bad
assert ok, "fix the cells above before unblinding"
print("\ncontract OK — the next stage UNBLINDS targeted-factor values.")

## Stage 5 — confirmatory analysis (**UNBLINDS**)

First real H1-H4 verdicts. The primary invariance rule is the **A10 §c two-sided deficit**
(`suppressed` iff `D = R(random) - R(trained) > epsilon_D`); the frozen one-sided boolean and both
frozen flip variants are co-reported as sensitivities under `*_frozen` keys. Quote them together,
never one alone.

In [ ]:
sh("python -m src.probes.hypotheses --root results/probes "
   "--out results/probes/hypotheses.json")

In [ ]:
try:
    sh("python -m src.eval.figures --root results/probes --out results/figures")
except SystemExit as e:
    print("figures failed (non-blocking, they carry no verdict):", e)

## Stage 6 — displacement spectrum, random + pixel (BLIND-SAFE, A11)

The null the destruction certificate is measured against. Needs >= 2 random seeds or `epsilon_m`
is undefined. Cheap: nothing is trained, it is forward passes over counterfactual pairs.

In [ ]:
from pathlib import Path

# --no-pixel-reference here too: the 12288-d whitener needs a 12288x12288 eigendecomposition
# (~1.2 GB float64 plus a 3.9 GB feature array) and is the pipeline's likeliest OOM. The
# destruction certificate's null is random-vs-random; pixels are A4 (b) context only.
for ds, cell in (("shapes3d", "reference_shapes3d"), ("dsprites", "reference_dsprites")):
    if Path(f"results/displacement/{cell}.json").exists():
        print(f"stage 6 [{cell}]: SKIP (exists)"); continue
    sh(f"python -m src.probes.displacement --dataset {ds} --cell {cell} "
       f"--random-seed {SEEDS} --n-contexts 512 --no-pixel-reference "
       f"--device cuda --num-workers 2 --out-root results/displacement")

## Stage 6b — ground-truth destruction control + external encoder (A13 c/d, BLIND-SAFE)

**The control that validates the certificate.** Greyscaling the input removes hue from the
*image*, so every hue factor is absent **by construction, independently of any encoder**.
`m_total` on the hue factors must collapse. This is the only case in the study where the right
answer is known without measurement — if the certificate does not fire here it is broken, and
that gets reported.

**The external check.** The same counterfactual pairs through an ImageNet-pretrained backbone:
exact pairs preserved, encoder entirely changed. Diagnostic only — its augmentation recipe is not
ours, so it licenses no invariance claim, only a test of whether the geometry is an artifact of
this study's training pipeline.

In [ ]:
from pathlib import Path

# A13 (c): hue is provably absent from a greyscale image -> m_total MUST collapse.
if not Path("results/displacement/groundtruth_grayscale.json").exists():
    sh(f"python -m src.probes.displacement --dataset shapes3d "
       f"--cell groundtruth_grayscale --grayscale --random-seed {SEEDS} "
       f"--n-contexts 512 --no-pixel-reference --device cuda --num-workers 2 "
       f"--out-root results/displacement")

# A13 (d): same pairs, ImageNet-pretrained encoder.
if not Path("results/displacement/external_shapes3d.json").exists():
    sh("python -m src.probes.displacement --dataset shapes3d "
       "--cell external_shapes3d --external-encoder resnet18 --random-seed 0 1 2 "
       "--n-contexts 512 --no-pixel-reference --device cuda --num-workers 2 "
       "--out-root results/displacement")

In [ ]:
import json
from pathlib import Path
import numpy as np

gt = Path("results/displacement/groundtruth_grayscale.json")
ref = Path("results/displacement/reference_shapes3d.json")
if gt.exists() and ref.exists():
    g, r = json.loads(gt.read_text()), json.loads(ref.read_text())
    print(f"{'factor':14s}{'m_total colour':>16s}{'m_total greyscale':>19s}{'ratio':>9s}")
    ok = True
    for f in next(iter(g["roles"]["random"].values())):
        mg = float(np.mean([g["roles"]["random"][t][f]["m_total"] for t in g["roles"]["random"]]))
        mr = float(np.mean([r["roles"]["random"][t][f]["m_total"] for t in r["roles"]["random"]]))
        ratio = mg / mr if mr > 0 else float("nan")
        hue = "hue" in f
        flag = "  <- HUE: must collapse" if hue else ""
        print(f"{f:14s}{mr:>16.4f}{mg:>19.4f}{ratio:>9.3f}{flag}")
        if hue and ratio > 0.25:
            ok = False
    print("\nA13 (c) GROUND-TRUTH CONTROL:",
          "PASS — hue displacement collapses when hue is removed from the input" if ok else
          "FAIL — the certificate does NOT fire where the factor is provably absent. "
          "Report this as a failure of the instrument; do not proceed to claims that rest on it.")

## Stage 7 — displacement on trained encoders + the A11 (d) link test (**UNBLINDS**)

Gated on A11 being in the frozen prereg — the cell checks the file rather than trusting a comment.
Preregistered target: Spearman `rho_s >= 0.5` with a bootstrap CI excluding 0, in the link test's
**own** Holm family. A CI excluding 0 below 0.5 is *detected but immaterial*, not a confirmation.
A refutation is reported as a refutation; the destruction certificate stands alone either way.

In [ ]:
from pathlib import Path

AMENDMENT = "A11-2026-08-13"
assert "### A11 " in Path("preregistration/prereg.md").read_text(), (
    "A11 is not in the frozen prereg — file it BEFORE computing rho_F on any trained encoder")

for cell, ds, _, pat in CELLS:
    if Path(f"results/displacement/{cell}.json").exists():
        print(f"stage 7 [{cell}]: SKIP (exists)"); continue
    sh(f"python -m src.probes.displacement --dataset {ds} --cell {cell} "
       f"--amendment {AMENDMENT} --trained-encoders {pat} --random-seed {SEEDS} "
       f"--n-contexts 512 --no-pixel-reference --device cuda --num-workers 2 --out-root results/displacement")

In [ ]:
import json
from pathlib import Path
import numpy as np
from scipy.stats import spearmanr

rows = []
for cell, *_ in CELLS:
    dp, hp = Path(f"results/displacement/{cell}.json"), Path(f"results/probes/{cell}/hypothesis_report.json")
    if not (dp.exists() and hp.exists()):
        print(f"{cell}: missing {'displacement' if not dp.exists() else 'hypothesis_report'}"); continue
    dj, hj = json.loads(dp.read_text()), json.loads(hp.read_text())
    tr, rnd, eps_m = dj["roles"].get("trained", {}), dj["roles"].get("random", {}), dj.get("epsilon_m", {})
    dg  = {r["factor"]: r["delta_g"] for r in hj["h1"]["per_factor"]}
    sat = set(hj.get("null_saturation", {}).get("saturated_factors_flip", []))
    diag = set(hj.get("report", {}).get("diagnostic_only_factors", []))
    for f in dg:
        if not tr or f not in next(iter(tr.values()), {}):
            continue
        m_tr = float(np.mean([tr[t][f]["m"] for t in tr]))
        m_rn = float(np.mean([rnd[t][f]["m"] for t in rnd])) if rnd else float("nan")
        e = eps_m.get(f, float("nan"))
        rows.append(dict(cell=cell, factor=f, m_trained=m_tr, m_random=m_rn,
                         deficit=m_rn - m_tr, epsilon_m=e,
                         destroyed=bool((m_rn - m_tr) > e) if e == e else None,
                         rho=float(np.mean([tr[t][f]["rho"] for t in tr])),
                         mono=float(np.mean([tr[t][f]["monotonicity"]["monotone_fraction"] for t in tr])),
                         delta_g=dg[f], eligible=f not in sat and f not in diag))

print(f"{'cell':16s}{'factor':14s}{'m(tr)':>9s}{'m(rnd)':>9s}{'deficit':>9s}{'eps_m':>8s}"
      f"{'destr':>7s}{'rho':>7s}{'mono':>7s}{'Delta_G':>9s}{'elig':>6s}")
for r in rows:
    print(f"{r['cell']:16s}{r['factor']:14s}{r['m_trained']:>9.4f}{r['m_random']:>9.4f}"
          f"{r['deficit']:>9.4f}{r['epsilon_m']:>8.4f}{str(r['destroyed']):>7s}"
          f"{r['rho']:>7.3f}{r['mono']:>7.2f}{r['delta_g']:>9.4f}{str(r['eligible']):>6s}")

elig = [r for r in rows if r["eligible"]]
if len(elig) >= 4:
    x = np.array([1.0 - r["rho_shared"] for r in elig])   # A12; y = np.array([r["delta_g"] for r in elig])
    rs = spearmanr(x, y).statistic
    rng = np.random.default_rng(0)
    boot = [spearmanr(x[i], y[i]).statistic for i in (rng.integers(0, len(x), len(x)) for _ in range(2000))]
    lo, hi = np.nanpercentile(boot, [2.5, 97.5])
    print(f"\nA11 (d)/A12 (c) LINK TEST  [(1 - rho_shared) -> Delta_G]: rho_s = {rs:+.3f}  CI95 [{lo:+.3f}, {hi:+.3f}]  n = {per_seed.size or len(elig)}")
    if rs >= 0.5 and (lo > 0 or hi < 0):
        print("VERDICT: CONFIRMED at the preregistered target. Report as consistent-with, "
              "never as established: this is one test on a three-cell grid (A11 e).")
    elif lo > 0 or hi < 0:
        print("VERDICT: association DETECTED BUT IMMATERIAL (CI excludes 0, rho_s < 0.5). "
              "Preregistered as NOT a confirmation.")
    else:
        print("VERDICT: REFUTED at the preregistered target. Report as a refutation — the "
              "destruction certificate (m_F vs epsilon_m) stands alone.")
else:
    print(f"\nA11 (d) link test not evaluable: {len(elig)} eligible readout(s), need >= 4.")

## Persist

`/kaggle/working` is the notebook output. Click **Save Version**, then **Add Input -> this output**
next run and this notebook resumes from wherever it stopped.

In [ ]:
import shutil
from pathlib import Path
src = Path("/kaggle/working/probe-capacity-invariance/results"); dst = Path("/kaggle/working/results")
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
print(f"persisted {sum(1 for _ in dst.rglob('*') if _.is_file())} files -> click 'Save Version'")